# Notebook 01


## Phase 1 — Load, Explore & Clean
### Python Foundations + Data Cleaning
### Your first job is to load the data, understand what each column means, and clean up the mess. By the end of this phase you should have a clean DataFrame with no obvious problems.
### Tasks (create notebook: 01_cleaning.ipynb)
• Load the data using pd.read_csv() and print the first 5 rows

• Check the shape: how many rows and columns?

• Inspect data types: use .info() — are any columns the wrong type? Fix at least 2

• Find missing values: use .isnull().sum() — which columns have the most? Decide what to
do (fill in or drop) and explain why

• Handle duplicates: check with .duplicated().sum() and remove if any

• Spot outliers: use a boxplot or the IQR method on the target column; cap extreme values at the
99th percentile

• Write a clean_data() function that does all the above steps in order so you can reuse it

• Add 3 checks at the end (e.g., no nulls in key columns, all target values > 0, correct number of
columns)


Tip: Add a comment or markdown cell explaining every decision. Future-you will thank you!

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/raw/AmesHousing.csv") # read dataset
df.head() # display first 5 rows
# • Load the data using pd.read_csv() and print the first 5 rows

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
df.shape # (rows, columns)
# • Check the shape: how many rows and columns?

(2930, 82)

In [4]:
cols = ["Bsmt Full Bath", "Bsmt Half Bath"] # columns to be updated

# show info before changes
print("BEFORE:")
df[cols].info()

# fill nulls and convert to int
df["Bsmt Full Bath"] = df["Bsmt Full Bath"].fillna(0).astype("int64")
df["Bsmt Half Bath"] = df["Bsmt Half Bath"].fillna(0).astype("int64")

# show info after changes
print("\n\nAFTER:")
df[cols].info()
# • Inspect data types: use .info() — are any columns the wrong type? Fix at least 2

BEFORE:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Bsmt Full Bath  2928 non-null   float64
 1   Bsmt Half Bath  2928 non-null   float64
dtypes: float64(2)
memory usage: 45.9 KB


AFTER:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Bsmt Full Bath  2930 non-null   int64
 1   Bsmt Half Bath  2930 non-null   int64
dtypes: int64(2)
memory usage: 45.9 KB


In [5]:
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)) # show missing values sorted in descending order

# Pool QC: Fill "None" because no pool exists
df["Pool QC"] = df["Pool QC"].fillna("None")

# Misc Feature: Fill "None" because no extra features
df["Misc Feature"] = df["Misc Feature"].fillna("None")

# Lot Frontage: Fill Median as it is safer for outliers
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

# Garage Type: Fill "None" because there is no garage
df["Garage Type"] = df["Garage Type"].fillna("None")

# Garage Yr Blt: Fill 0 to show no garage exists
df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(0)

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Yr Blt      159
Garage Finish      159
Garage Cond        159
Garage Qual        159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Qual           80
Bsmt Cond           80
BsmtFin Type 1      80
Mas Vnr Area        23
BsmtFin SF 1         1
Bsmt Unf SF          1
Total Bsmt SF        1
Electrical           1
BsmtFin SF 2         1
Garage Area          1
Garage Cars          1
dtype: int64


In [6]:
print(df.duplicated().sum()) # count duplicates
df = df.drop_duplicates() # remove duplicates
# No duplicates (0)
# • Handle duplicates: check with .duplicated().sum() and remove if any

0


In [7]:
q1 = df["SalePrice"].quantile(0.25) # calculate first quartile
q3 = df["SalePrice"].quantile(0.75) # calculate third quartile
iqr = q3 - q1 # iqr formula

lower_limit = q1 - 1.5 * iqr # lower limit
upper_limit = q3 + 1.5 * iqr # upper limit

outliers = ((df["SalePrice"] < lower_limit) | (df["SalePrice"] > upper_limit)).sum() # count outliers
print(f"outliers: {outliers}") # show total

p99 = df["SalePrice"].quantile(0.99) # get the 99th
df["SalePrice"] = df["SalePrice"].clip(upper=p99) # cap at 99th
# • Spot outliers: use a boxplot or the IQR method on the target column; cap extreme values at the 99th percentile

outliers: 137


In [8]:
def clean_data(data: pd.DataFrame) -> pd.DataFrame:
    print(data.head())
    print(data.shape)
    data["Bsmt Full Bath"] = data["Bsmt Full Bath"].fillna(0).astype("int64")
    data["Bsmt Half Bath"] = data["Bsmt Half Bath"].fillna(0).astype("int64")
    data["Pool QC"] = data["Pool QC"].fillna("None")
    data["Misc Feature"] = data["Misc Feature"].fillna("None")
    data["Lot Frontage"] = data["Lot Frontage"].fillna(data["Lot Frontage"].median())
    data["Garage Type"] = data["Garage Type"].fillna("None")
    data["Garage Yr Blt"] = data["Garage Yr Blt"].fillna(0)
    print(f"duplicated: {data.duplicated().sum()}")
    data = data.drop_duplicates()
    q1 = data["SalePrice"].quantile(0.25)
    q3 = data["SalePrice"].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outliers = ((data["SalePrice"] < lower_limit) | (data["SalePrice"] > upper_limit)).sum()
    print(f"outliers: {outliers}")
    q99 = data["SalePrice"].quantile(0.99)
    data["SalePrice"] = data["SalePrice"].clip(upper=q99)
    data.info()
    return data
# • Write a clean_data() function that does all the above steps in order so you can reuse it

In [9]:
clean_data(df)

   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0    None    NaN         None   
1   NaN       Reg          Lvl  ...         0    None  MnPrv         None   
2   NaN       IR1          Lvl  ...         0    None    NaN         Gar2   
3   NaN       Reg          Lvl  ...         0    None    NaN         None   
4   NaN       IR1          Lvl  ...         0    None  MnPrv         None   

  Misc Val Mo Sold Yr Sold Sale Type  Sale Condition  SalePrice  
0       

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,None,NaN,None,0,5,2010,WD,Normal,215000.0
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000.0
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,None,NaN,Gar2,12500,6,2010,WD,Normal,172000.0
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,None,NaN,None,0,4,2010,WD,Normal,244000.0
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,None,GdPrv,None,0,3,2006,WD,Normal,142500.0
2926,2927,923276100,20,RL,68.0,8885,Pave,NaN,IR1,Low,...,0,None,MnPrv,None,0,6,2006,WD,Normal,131000.0
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,None,MnPrv,Shed,700,7,2006,WD,Normal,132000.0
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,None,NaN,None,0,4,2006,WD,Normal,170000.0


In [10]:
def check_data(data: pd.DataFrame) -> pd.DataFrame:
    assert data[["SalePrice", "Overall Qual", "Gr Liv Area"]].isnull().sum().sum() == 0
    assert (data["SalePrice"] > 0).all()
    assert data.shape[1] == 82

In [11]:
check_data(df)